# 05 - Object Detection (single video)

YOLO11m detection on keyframes generated by stage 01. This notebook stores boxes, classes, confidence, and exact frame metadata.


In [ ]:
%pip install -q ultralytics pandas pyarrow pillow matplotlib


In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

VIDEO_ID = "L21_V001"
PIPELINE_VERSION = "aicv3-object-yolo11m"
MODEL_NAME = "yolo11m.pt"
CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.70
IMAGE_SIZE = 960
BATCH_SIZE = 16

matches = sorted(Path("/kaggle/input").rglob("keyframes.parquet"))
if not matches:
    raise FileNotFoundError("No keyframes.parquet found in /kaggle/input")

KEYFRAMES_PATH = next((path for path in matches if VIDEO_ID in path.as_posix()), matches[0])
INPUT_ROOT = KEYFRAMES_PATH.parent
FRAME_ROOTS = list(INPUT_ROOT.rglob("frames"))
FRAME_ROOT = FRAME_ROOTS[0] if FRAME_ROOTS else INPUT_ROOT / "frames"
OUTPUT_ROOT = Path("/kaggle/working/05-object-yolo11m-output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

keyframes_df = pd.read_parquet(KEYFRAMES_PATH)
if "video_id" in keyframes_df.columns:
    keyframes_df = keyframes_df[keyframes_df.video_id.astype(str) == VIDEO_ID].copy()
keyframes_df = keyframes_df.sort_values("frame_idx").reset_index(drop=True)
required = {"keyframe_id", "video_id", "frame_idx", "pts_time", "frame_path"}
missing = required - set(keyframes_df.columns)
if missing:
    raise ValueError(f"Missing keyframe columns: {sorted(missing)}")
if keyframes_df.empty:
    raise ValueError(f"No keyframes found for {VIDEO_ID}")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Video:", VIDEO_ID)
print("Keyframes:", len(keyframes_df))
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")


In [ ]:
def resolve_frame_path(value: str) -> Path:
    path = INPUT_ROOT / value
    if path.exists():
        return path
    path = FRAME_ROOT.parent / value
    if path.exists():
        return path
    candidates = list(INPUT_ROOT.rglob(Path(value).name))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(value)

model = YOLO(MODEL_NAME)
print("YOLO ready:", MODEL_NAME)


In [ ]:
records = []
errors = []
paths = []
metadata_rows = []
for row in keyframes_df.itertuples(index=False):
    try:
        path = resolve_frame_path(str(row.frame_path))
        paths.append(str(path))
        metadata_rows.append(row)
    except Exception as exc:
        errors.append({"keyframe_id": str(row.keyframe_id), "error": str(exc)})

for start in range(0, len(paths), BATCH_SIZE):
    batch_paths = paths[start:start + BATCH_SIZE]
    batch_meta = metadata_rows[start:start + BATCH_SIZE]
    results = model.predict(
        source=batch_paths,
        conf=CONFIDENCE_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        verbose=False,
    )
    for row, result, frame_path in zip(batch_meta, results, batch_paths):
        boxes = result.boxes
        names = result.names
        if boxes is None or len(boxes) == 0:
            continue
        xyxy = boxes.xyxy.detach().cpu().numpy()
        confs = boxes.conf.detach().cpu().numpy()
        classes = boxes.cls.detach().cpu().numpy().astype(int)
        height, width = Image.open(frame_path).size[1], Image.open(frame_path).size[0]
        for box, confidence, class_id in zip(xyxy, confs, classes):
            x1, y1, x2, y2 = [float(value) for value in box]
            records.append({
                "detection_id": f"{row.keyframe_id}:det:{len(records):06d}",
                "keyframe_id": str(row.keyframe_id),
                "video_id": str(row.video_id),
                "frame_idx": int(row.frame_idx),
                "pts_time": float(row.pts_time),
                "frame_path": str(row.frame_path),
                "class_id": int(class_id),
                "class_name": str(names[int(class_id)]),
                "confidence": float(confidence),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": int(width),
                "height": int(height),
                "bbox_norm": [x1 / width, y1 / height, x2 / width, y2 / height],
            })
    print(f"Detection: {min(start + len(batch_paths), len(paths))}/{len(paths)} frames")

objects_df = pd.DataFrame(records)
if objects_df.empty:
    raise RuntimeError("YOLO produced no detections")
objects_df.to_parquet(OUTPUT_ROOT / "objects.parquet", index=False)
objects_df.to_json(OUTPUT_ROOT / "objects.jsonl", orient="records", lines=True, force_ascii=False)
summary = {
    "stage": "object-detection",
    "pipeline_version": PIPELINE_VERSION,
    "video_id": VIDEO_ID,
    "model": MODEL_NAME,
    "keyframes": int(len(keyframes_df)),
    "frames_read": int(len(paths)),
    "frame_errors": int(len(errors)),
    "detections": int(len(objects_df)),
    "unique_frames_with_objects": int(objects_df.frame_idx.nunique()),
    "classes": sorted(objects_df.class_name.unique().tolist()),
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "image_size": IMAGE_SIZE,
}
(OUTPUT_ROOT / "object-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_ROOT / "object-errors.json").write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Top classes:")
display(objects_df.groupby("class_name").size().sort_values(ascending=False).head(20).rename("detections").to_frame())


In [ ]:
# Preview detections on up to 12 frames.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

sample_ids = objects_df.keyframe_id.drop_duplicates().head(12).tolist()
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for axis in axes.flat:
    axis.axis("off")
for axis, keyframe_id in zip(axes.flat, sample_ids):
    row = keyframes_df[keyframes_df.keyframe_id == keyframe_id].iloc[0]
    image_path = resolve_frame_path(str(row.frame_path))
    image = Image.open(image_path).convert("RGB")
    axis.imshow(image)
    axis.set_title(f"{VIDEO_ID} frame {row.frame_idx}", fontsize=9)
    for item in objects_df[objects_df.keyframe_id == keyframe_id].itertuples():
        axis.add_patch(Rectangle((item.x1, item.y1), item.x2-item.x1, item.y2-item.y1, fill=False, linewidth=1.5, edgecolor="red"))
        axis.text(item.x1, max(0, item.y1 - 3), f"{item.class_name} {item.confidence:.2f}", color="white", fontsize=7, backgroundcolor="red")
plt.tight_layout()
plt.show()


In [ ]:
archive = shutil.make_archive(
    f"/kaggle/working/05-object-yolo11m-{VIDEO_ID}",
    "zip",
    root_dir=OUTPUT_ROOT,
)
print("Download:", archive)
